# CV Masterclass 08: Generative Vision Lineage

Welcome to Notebook 08. This notebook traces the complete historical and mathematical lineage of **Generative Computer Vision**.

We do not jump straight to the state-of-the-art. To understand *why* Diffusion Models exist, you must understand the mathematical collapse of GANs. To understand GANs, you must understand the blurriness of VAEs. To understand VAEs, you must understand the deterministic failure of standard Autoencoders.

---

## 🎓 Socratic Deep Check
Ponder this question as we progress through the lineage:

> *"A standard Autoencoder compresses a dog into a latent vector `[2.5, -1.0]`, and successfully reconstructs it. But if you pass the random vector `[2.4, -0.9]` into the Decoder, it outputs absolute garbage instead of a slightly different dog. Why does the math of a Variational Autoencoder (VAE) specifically fix this, allowing us to safely generate new images from random vectors?"*

---

## Table of Contents
1. **The Foundation:** Standard Autoencoders (Compression vs Generation).
2. **The Probabilistic Shift:** Variational Autoencoders (VAEs) & The Reparameterization Trick.
3. **The Adversarial Era:** GANs, Minimax Games, and catastrophic Mode Collapse.
4. **The Thermodynamic Override:** Diffusion Models (DDPMs) and stable Forward/Reverse Markov Chains.


## 1. Standard Autoencoders (The Deterministic Bottleneck)

An Autoencoder (AE) has two parts:
1.  **Encoder:** Compresses an $H \times W \times C$ image into a tiny 1-Dimensional Latent Vector (e.g., $Z$, length 128).
2.  **Decoder:** Expands the 128-dim vector back into the original $H \times W \times C$ image.

**Loss:** Mean Squared Error (MSE) between the input image and the output image.

**The Generative Failure:** AEs are deterministic. The network learns to map exactly 1 specific training image to exactly 1 specific point in the 128-D space. The space between the points is completely "empty" or discontinuous. If you sample a random point in that space (trying to *generate* a new image), the decoder panics and outputs meaningless noise. AEs are purely for compression, not generation.

## 2. Variational Autoencoders (VAEs)

A VAE forces the Latent Space to be continuous. 

Instead of the Encoder outputting a single point $Z = [2.5, -1.0]$, it outputs two vectors:
1.  **Mean ($\mu$):** The center of a distribution.
2.  **Standard Deviation ($\sigma$):** The spread of the distribution.

We then sample a random point from $\mathcal{N}(\mu, \sigma^2)$ and pass *that* to the Decoder. 

Because the decoder is constantly being fed slightly different points from the distribution during training, it is forced to learn that *entire region* corresponds to a "Dog". The space becomes perfectly smooth and continuous.

### The Reparameterization Trick
If you have a Python operation `z = np.random.normal(mu, sigma)`, PyTorch cannot backpropagate through `random()`. The computational graph breaks.
**The Math Fix:** We sample from a standard Normal Distribution $\epsilon \sim \mathcal{N}(0, 1)$ (which requires no gradients), and mathematically shift it: $Z = \mu + \sigma \odot \epsilon$. Now, Backprop flows perfectly through $\mu$ and $\sigma$!

In [ ]:
import torch
import torch.nn as nn

# -----------------------------------------------------
# IMPLEMENTATION: The VAE Reparameterization Trick
# -----------------------------------------------------
class VAE_Bottleneck(nn.Module):
    def __init__(self, in_features=512, latent_dim=128):
        super(VAE_Bottleneck, self).__init__()
        # The Encoder spits out Mu and Log-Variance (Log is used for numerical stability)
        self.fc_mu = nn.Linear(in_features, latent_dim)
        self.fc_logvar = nn.Linear(in_features, latent_dim)
        
    def reparameterize(self, mu, logvar):
        # 1. Convert Log-Variance back to Standard Deviation
        std = torch.exp(0.5 * logvar)
        
        # 2. Sample random Epsilon from standard Normal N(0, 1)
        # This operation has NO gradients attached to it.
        eps = torch.randn_like(std)
        
        # 3. Shift and Scale (The Reparameterization Trick)
        # Gradients can now flow directly backwards into 'mu' and 'std'
        z = mu + (std * eps)
        return z
        
    def forward(self, x):
        mu = self.fc_mu(x)
        logvar = self.fc_logvar(x)
        z = self.reparameterize(mu, logvar)
        return z, mu, logvar

print("VAE Reparameterization logic defined. The graph survives the random sampling.")
# VAE Flaw: Because they rely on Mean Squared Error pixel-by-pixel comparisons, VAE images are notoriously BLURRY.

## 3. Generative Adversarial Networks (GANs)

Because MSE in VAEs averages out uncertainty, VAEs produce blurry images. 

In 2014, Ian Goodfellow invented the GAN. We delete the MSE loss entirely. We implement a **Zero-Sum Minimax Game**:
1.  **Generator ($G$):** Takes a random noise vector $Z$ and tries to create crisp, photorealistic pixels to fool the Discriminator.
2.  **Discriminator ($D$):** A binary classifier trying to tell Real images from Fake ($G$) images.

$\min_G \max_D V(D, G) = \mathbb{E}_{x}[\log D(x)] + \mathbb{E}_{z}[\log(1 - D(G(z)))]$

**The Catastrophic Flaw: Mode Collapse**
If $G$ discovers it generates highly realistic "Cats", $D$ will classify them as "Real". Because $G$'s only objective is to fool $D$, $G$ stops trying to generate Dogs or Cars entirely. It "collapses" into a single visual mode. 

Furthermore, tracking gradients across two dueling networks is mathematically chaotic. If $D$ gets too good too fast, its gradients drop to $0.0$, and $G$ stops learning completely.

In [ ]:
import torch.nn.functional as F

# -----------------------------------------------------
# IMPLEMENTATION: The WGAN Gradient Penalty (Fixing GANs)
# -----------------------------------------------------
"""
To fix GAN instability, researchers invented Wasserstein GANs (WGAN).
It requires the Discriminator (now called a Critic) to be smooth (1-Lipschitz continuous). 
The modern way to enforce this is mathematically adding a 'Gradient Penalty' to the loss function.
"""
def compute_gradient_penalty(critic, real_samples, fake_samples):
    """Calculates the mathematically enforced Gradient Penalty for WGAN-GP."""
    # 1. Random weight term for interpolation between real and fake samples
    alpha = torch.rand((real_samples.size(0), 1, 1, 1), device=real_samples.device)
    
    # 2. Get random interpolation between real and fake samples (The space between)
    interpolates = (alpha * real_samples + ((1 - alpha) * fake_samples)).requires_grad_(True)
    
    # 3. Calculate Critic scores
    d_interpolates = critic(interpolates)
    
    # 4. Extract explicit gradients of the Critic with respect to the interpolates
    fake = torch.ones(d_interpolates.shape, requires_grad=False, device=real_samples.device)
    gradients = torch.autograd.grad(
        outputs=d_interpolates,
        inputs=interpolates,
        grad_outputs=fake,
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]
    
    # 5. Penalize any gradient whose L2 norm deviates from 1.0
    gradients = gradients.view(gradients.size(0), -1)
    gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
    return gradient_penalty

print("WGAN Gradient Penalty defined. This stabilized GANs until Diffusion arrived.")

## 4. Diffusion Models (The Thermodynamic Override)

GANs generate images in a single, explosive, unstable shot. 

**Diffusion Models (DDPMs)** abandoned adversarial games for Information Theory.

1.  **Forward Process (Markov Chain):** You take an image $X_0$. Over $T=1000$ tiny steps, you mathematically inject Gaussian noise until it is pure static $X_T$. Because this is a fixed mathematical formula, it requires no Neural Network. It is perfectly stable.
2.  **Reverse Process (Denoising U-Net):** You train a U-Net to take $X_t$ and simply predict the tiny noise vector you just added. 

**Why this scales to Billions of Parameters:**
The U-Net doesn't have to guess what the final image looks like. It is strictly predicting the physical noise distribution at step $t$. This is a simple L2 Mean Squared Error regression task. The loss landscape is beautiful, highly convex, and completely immune to Mode Collapse. 

By dividing the generation task into 1,000 microscopic baby steps, Diffusion models conquered the world (Midjourney, Stable Diffusion, DALL-E).